# SMD Data Viewer

Every row = one timestep &nbsp;|&nbsp; All 38 sensor columns &nbsp;|&nbsp; Anomaly rows highlighted in light red

| Column | Meaning |
|--------|---------|
| `timestamp` | 1-based row index |
| `cpu_r` … `udp_snd_buf_errs` | 38 sensor features (normalised 0–1) |
| `label` | 0 = Normal, 1 = Anomaly |
| `machine_id` | Source machine |

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

PROJECT_ROOT  = Path('../../').resolve()
ALL_TEST_CSV  = PROJECT_ROOT / 'data' / 'processed' / 'all_test.csv'
ALL_TRAIN_CSV = PROJECT_ROOT / 'data' / 'processed' / 'all_train.csv'

FEATURE_NAMES = [
    'cpu_r','load_1','load_5','load_15','mem_shmem','mem_u','mem_u_e',
    'total_mem','disk_q','disk_r','disk_rb','disk_svc','disk_u','disk_w',
    'disk_wa','disk_wb','si','so','eth1_fi','eth1_fo','eth1_pi','eth1_po',
    'tcp_tw','tcp_use','active_opens','curr_estab','in_errs','in_segs',
    'listen_overflows','out_rsts','out_segs','passive_opens','retransegs',
    'tcp_timeouts','udp_in_dg','udp_out_dg','udp_rcv_buf_errs','udp_snd_buf_errs',
]

# pandas >=2.1 renamed applymap -> map
_pd_ver   = tuple(int(x) for x in pd.__version__.split('.')[:2])
_stylemap = 'map' if _pd_ver >= (2, 1) else 'applymap'

print(f'Pandas {pd.__version__} | Test: {ALL_TEST_CSV.exists()} | Train: {ALL_TRAIN_CSV.exists()}')

Pandas 3.0.5 | Test: True | Train: True


In [2]:
# Load processed CSVs
test_df  = pd.read_csv(ALL_TEST_CSV)
train_df = pd.read_csv(ALL_TRAIN_CSV)

# 1-based timestamp column
test_df.insert(0,  'timestamp', range(1, len(test_df)  + 1))
train_df.insert(0, 'timestamp', range(1, len(train_df) + 1))

print(f'Test  : {test_df.shape[0]:,} rows x {test_df.shape[1]} cols')
print(f'Train : {train_df.shape[0]:,} rows x {train_df.shape[1]} cols')

Test  : 679,698 rows x 41 cols
Train : 679,683 rows x 40 cols


---
### Settings

In [3]:
MACHINE   = 'machine-1-1'   # machine to view
SPLIT     = 'test'          # 'test' or 'train'
FILTER    = 'all'           # 'all' | 'normal' | 'anomaly'
N_ROWS    = 200             # rows to show
PRECISION = 4               # decimal places

print(f'Machine={MACHINE}  Split={SPLIT}  Filter={FILTER}  Rows={N_ROWS}')

Machine=machine-1-1  Split=test  Filter=all  Rows=200


In [4]:
# ── Slice & filter ────────────────────────────────────────────────────────────
source = test_df if SPLIT == 'test' else train_df
view   = source[source['machine_id'] == MACHINE].copy().reset_index(drop=True)

if FILTER == 'normal'  and SPLIT == 'test': view = view[view['label'] == 0]
if FILTER == 'anomaly' and SPLIT == 'test': view = view[view['label'] == 1]

view['timestamp'] = range(1, len(view) + 1)

cols = (['timestamp'] + FEATURE_NAMES + ['label', 'machine_id']
        if SPLIT == 'test' else
        ['timestamp'] + FEATURE_NAMES + ['machine_id'])
view = view[cols].head(N_ROWS)

n_anom = int(view['label'].sum()) if SPLIT == 'test' else 0
print(f'Showing {len(view):,} rows | Anomalies in view: {n_anom} ({n_anom/max(len(view),1)*100:.1f}%)')

Showing 200 rows | Anomalies in view: 0 (0.0%)


In [5]:
# ── Render table ──────────────────────────────────────────────────────────────

def row_highlight(row):
    """Light red for anomaly rows, plain white for normal."""
    if SPLIT == 'test' and 'label' in row.index and row['label'] == 1:
        return ['background-color: #fde8e8'] * len(row)
    return [''] * len(row)

def label_colour(val):
    """Colour the label cell text only."""
    if val == 1: return 'color: #c0392b; font-weight: bold'
    if val == 0: return 'color: #27ae60; font-weight: bold'
    return ''

# Number format for sensor columns
fmt = {col: f'{{:.{PRECISION}f}}' for col in FEATURE_NAMES if col in view.columns}

TABLE_STYLES = [
    # Header — light grey, normal dark text
    {'selector': 'thead th',
     'props': [('background-color', '#f0f0f0'),
               ('color', '#222222'),
               ('font-size', '11px'),
               ('font-family', 'sans-serif'),
               ('padding', '6px 10px'),
               ('text-align', 'center'),
               ('white-space', 'nowrap'),
               ('border-bottom', '2px solid #cccccc'),
               ('position', 'sticky'),
               ('top', '0'),
               ('z-index', '2')]},
    # Data cells
    {'selector': 'td',
     'props': [('padding', '4px 9px'),
               ('font-size', '11px'),
               ('font-family', 'monospace'),
               ('text-align', 'right'),
               ('white-space', 'nowrap'),
               ('border-bottom', '1px solid #eeeeee'),
               ('color', '#333333')]},
    # Timestamp column
    {'selector': 'td:first-child',
     'props': [('background-color', '#f7f7f7'),
               ('color', '#555555'),
               ('font-weight', 'bold'),
               ('text-align', 'center'),
               ('border-right', '1px solid #dddddd')]},
    # Alternating rows (subtle, overridden by anomaly red)
    {'selector': 'tbody tr:nth-child(even)',
     'props': [('background-color', '#fafafa')]},
    {'selector': 'table',
     'props': [('border-collapse', 'collapse'),
               ('width', '100%')]},
]

styler = (
    view.style
    .apply(row_highlight, axis=1)
    .format(fmt)
    .set_table_styles(TABLE_STYLES)
    .hide(axis='index')
)

if SPLIT == 'test' and 'label' in view.columns:
    styler = getattr(styler, _stylemap)(label_colour, subset=['label'])

html = styler.to_html()

legend = '<span style="color:#c0392b">&#9632;</span> Anomaly &nbsp; <span style="color:#27ae60">&#9632;</span> Normal' if SPLIT == 'test' else ''

display(HTML(f"""
<div style="font-family:sans-serif; margin-bottom:6px; font-size:13px; color:#333;">
  <strong>{MACHINE}</strong> &nbsp;|&nbsp; {SPLIT.upper()} split
  &nbsp;|&nbsp; filter: <code>{FILTER}</code>
  &nbsp;|&nbsp; {len(view):,} rows shown
  &nbsp;&nbsp; {legend}
</div>
<div style="overflow-x:auto; overflow-y:auto; max-height:550px;
            border:1px solid #dddddd; border-radius:6px; background:#ffffff;">
  {html}
</div>
"""))

timestamp,cpu_r,load_1,load_5,load_15,mem_shmem,mem_u,mem_u_e,total_mem,disk_q,disk_r,disk_rb,disk_svc,disk_u,disk_w,disk_wa,disk_wb,si,so,eth1_fi,eth1_fo,eth1_pi,eth1_po,tcp_tw,tcp_use,active_opens,curr_estab,in_errs,in_segs,listen_overflows,out_rsts,out_segs,passive_opens,retransegs,tcp_timeouts,udp_in_dg,udp_out_dg,udp_rcv_buf_errs,udp_snd_buf_errs,label,machine_id
1,0.0753,0.0657,0.0702,0.0743,0.0000,0.9333,0.2740,0.0000,0.0311,0.0000,0.1341,0.0811,0.0274,0.0678,0.1258,0.1506,0.0000,0.0000,0.1220,0.0920,0.0940,0.0742,0.9354,0.0181,0.0320,0.0166,0.0000,0.0777,0.0000,0.0086,0.0680,0.0489,0.0004,0.0000,0.0644,0.0645,0.0000,0.0000,0,machine-1-1
2,0.0860,0.0805,0.0758,0.0767,0.0000,0.9308,0.2750,0.0000,0.0311,0.0001,0.1488,0.1622,0.0548,0.0714,0.1231,0.1645,0.0000,0.0000,0.1214,0.0942,0.0962,0.0765,0.9966,0.0160,0.0336,0.0147,0.0000,0.0794,0.0000,0.0086,0.0700,0.0504,0.0004,0.0000,0.0652,0.0652,0.0000,0.0000,0,machine-1-1
3,0.0753,0.0646,0.0713,0.0743,0.0000,0.9282,0.2750,0.0000,0.0309,0.0004,0.1348,0.0946,0.0274,0.0633,0.1290,0.1515,0.0000,0.0000,0.1154,0.0901,0.0959,0.0757,0.9603,0.0118,0.0336,0.0109,0.0000,0.0795,0.0000,0.0086,0.0697,0.0551,0.0004,0.0000,0.0671,0.0672,0.0000,0.0000,0,machine-1-1
4,0.0860,0.0487,0.0635,0.0708,0.0000,0.9282,0.2731,0.0000,0.0272,0.0002,0.1313,0.0811,0.0274,0.0678,0.1104,0.1456,0.0000,0.0000,0.1351,0.0974,0.1014,0.0802,0.9615,0.0161,0.0360,0.0147,0.0000,0.0834,0.0000,0.0100,0.0733,0.0515,0.0000,0.0000,0.0667,0.0667,0.0000,0.0000,0,machine-1-1
5,0.0860,0.0519,0.0624,0.0708,0.0000,0.9333,0.2740,0.0000,0.0309,0.0002,0.1027,0.1081,0.0411,0.0757,0.1191,0.1184,0.0000,0.0000,0.1274,0.0945,0.0982,0.0775,0.9422,0.0184,0.0341,0.0170,0.0000,0.0809,0.0000,0.0086,0.0709,0.0515,0.0004,0.0000,0.0666,0.0667,0.0000,0.0000,0,machine-1-1
6,0.0860,0.0858,0.0691,0.0720,0.0000,0.9282,0.2731,0.0000,0.0308,0.0002,0.1651,0.0946,0.0274,0.0685,0.1243,0.1826,0.0000,0.0000,0.1262,0.0942,0.0997,0.0787,0.9531,0.0163,0.0353,0.0150,0.0000,0.0822,0.0000,0.0086,0.0720,0.0525,0.0004,0.0000,0.0674,0.0675,0.0000,0.0000,0,machine-1-1
7,0.0860,0.0710,0.0669,0.0708,0.0000,0.9256,0.2721,0.0000,0.0297,0.0006,0.1194,0.1351,0.0411,0.0672,0.1207,0.1346,0.0000,0.0000,0.1224,0.0929,0.0945,0.0745,0.9324,0.0139,0.0322,0.0127,0.0000,0.0782,0.0000,0.0086,0.0684,0.0479,0.0004,0.0000,0.0653,0.0654,0.0000,0.0000,0,machine-1-1
8,0.0860,0.0593,0.0635,0.0685,0.0000,0.9256,0.2702,0.0000,0.0315,0.0000,0.1347,0.0811,0.0274,0.0658,0.1292,0.1502,0.0000,0.0000,0.1213,0.0920,0.0943,0.0745,0.9504,0.0147,0.0329,0.0135,0.0000,0.0784,0.0000,0.0100,0.0686,0.0504,0.0004,0.0000,0.0646,0.0646,0.0000,0.0000,0,machine-1-1
9,0.0860,0.0434,0.0569,0.0662,0.0000,0.9282,0.2702,0.0000,0.0281,0.0000,0.1768,0.0676,0.0274,0.0660,0.1153,0.1908,0.0000,0.0000,0.1211,0.0927,0.0981,0.0775,0.9182,0.0152,0.0346,0.0141,0.0000,0.0810,0.0000,0.0086,0.0710,0.0535,0.0000,0.0000,0.0673,0.0673,0.0000,0.0000,0,machine-1-1
10,0.0860,0.0508,0.0557,0.0650,0.0000,0.9308,0.2702,0.0000,0.0319,0.0001,0.1779,0.1081,0.0411,0.0788,0.1209,0.1927,0.0000,0.0000,0.1200,0.0916,0.0953,0.0752,0.9314,0.0119,0.0324,0.0108,0.0000,0.0787,0.0000,0.0086,0.0690,0.0499,0.0004,0.0000,0.0659,0.0659,0.0000,0.0000,0,machine-1-1


---
### Label distribution

In [6]:
if SPLIT == 'test':
    mt = test_df[test_df['machine_id'] == MACHINE]
    n_total  = len(mt)
    n_anom   = int(mt['label'].sum())
    n_normal = n_total - n_anom

    dist_df = pd.DataFrame([
        {'Label': 0,   'Class': 'Normal',  'Count': n_normal, 'Pct (%)': round(n_normal/n_total*100, 2)},
        {'Label': 1,   'Class': 'Anomaly', 'Count': n_anom,   'Pct (%)': round(n_anom/n_total*100, 2)},
        {'Label': '-', 'Class': 'TOTAL',   'Count': n_total,  'Pct (%)': 100.0},
    ])

    DIST_STYLES = [
        {'selector': 'th',
         'props': [('background-color','#f0f0f0'), ('color','#222222'),
                   ('padding','8px 18px'), ('font-size','12px'),
                   ('font-family','sans-serif'), ('border-bottom','2px solid #cccccc')]},
        {'selector': 'td',
         'props': [('padding','6px 18px'), ('font-size','12px'),
                   ('font-family','sans-serif'), ('text-align','center'),
                   ('border-bottom','1px solid #eeeeee'), ('color','#333333')]},
        {'selector': 'tr:last-child td',
         'props': [('font-weight','bold'), ('background-color','#f5f5f5')]},
    ]

    styler2 = (
        dist_df.style
        .set_table_styles(DIST_STYLES)
        .format({'Count': '{:,}', 'Pct (%)': '{:.2f}'})
        .hide(axis='index')
        .bar(subset=['Pct (%)'], color='#a8d8a8', vmin=0, vmax=100)
    )

    def _lc(val):
        if val == 1: return 'color:#c0392b; font-weight:bold'
        if val == 0: return 'color:#27ae60; font-weight:bold'
        return ''

    styler2 = getattr(styler2, _stylemap)(_lc, subset=['Label'])
    display(styler2)
else:
    print('Train split has no labels (unsupervised — normal data only).')

Label,Class,Count,Pct (%)
0,Normal,"25,785",90.54
1,Anomaly,"2,694",9.46
-,TOTAL,"28,479",100.00
